# Fynd AI Intern – Task 1  
## Rating Prediction via Prompting (Yelp Reviews + Groq Llama-3.1-70B)

This notebook implements Task 1:

- Load and sample Yelp Reviews dataset  
- Use **3 prompting approaches** for rating prediction  
- Use **llama-3.1-8b-instant** for structured JSON predictions  
- Evaluate:
  - Accuracy  
  - JSON validity  
  - Reliability  
- Produce comparison tables + examples

> Note: Set `GROQ_API_KEY` before running.


## 1. Setup & Installation
Install Groq SDK and dependencies.


In [30]:
!pip install -q groq pandas tqdm

import os
import json
import random
import pandas as pd
from tqdm import tqdm
from groq import Groq
from google.colab import userdata # Import userdata to load from Colab secrets

# ======== Groq Configuration =========

# Option A (recommended): set before launching Jupyter:
#   export GROQ_API_KEY="your_key_here"
#
# Option B: uncomment and paste directly here (never commit):
# os.environ["GROQ_API_KEY"] = "YOUR_ACTUAL_GROQ_API_KEY_HERE" # <-- REPLACE THIS WITH YOUR KEY

# Option C: Load from Colab secrets (Recommended)
GROQ_API_KEY = userdata.get('GROQ_API_KEY') # Make sure you've stored your key as 'GROQ_API_KEY' in Colab secrets

if GROQ_API_KEY is None:
    raise ValueError(
        "Missing GROQ_API_KEY. Set it as an environment variable or in Colab secrets as 'GROQ_API_KEY'."
    )

client = Groq(api_key=GROQ_API_KEY)

# Updated model name from 'llama-3.1-70b-versatile' to a currently supported model.
# Refer to Groq's documentation for the latest available models.
# Changed from 'llama3-70b-8192' to 'llama-3.1-8b-instant' as 'llama3-70b-8192' is also decommissioned.
MODEL_NAME = "llama-3.1-8b-instant"

print("Groq model configured:", MODEL_NAME)

Groq model configured: llama-3.1-8b-instant


## 2. Load Yelp Dataset
Ensure `yelp_reviews.csv` is in the same folder.


In [22]:
DATA_PATH = "yelp.csv"

df = pd.read_csv(DATA_PATH)

print("Columns:", df.columns.tolist())
print("Rows:", len(df))
df.head()


Columns: ['business_id', 'date', 'review_id', 'stars', 'text', 'type', 'user_id', 'cool', 'useful', 'funny']
Rows: 10000


,business_id,date,review_id,stars,text,type,user_id,cool,useful,funny
0,9yKzy9PApeiPPOUJEtnvkg,2011-01-26,fWKvX83p0-ka4JS3dc6E5A,5,My wife took me here on my birthday for breakf...,review,rLtl8ZkDX5vH5nAx9C3q5Q,2,5,0
1,ZRJwVLyzEJq1VAihDhYiow,2011-07-27,IjZ33sJrzXqU-0X6U8NwyA,5,I have no idea why some people give bad review...,review,0a2KyEL0d3Yb1V6aivbIuQ,0,0,0
2,6oRAC4uyJCsJl1X0WZpVSA,2012-06-14,IESLBzqUCLdSzSqm0eCSxQ,4,love the gyro plate. Rice is so good and I als...,review,0hT2KtfLiobPvh6cDC8JQg,0,1,0
3,_1QQZuf4zZOyFCvXc0o6Vg,2010-05-27,G-WvGaISbqqaMHlNnByodA,5,"Rosie, Dakota, and I LOVE Chaparral Dog Park!!...",review,uZetl9T0NcROGOyFfughhg,1,2,0
4,6ozycU1RpktNG2-1BroVtw,2012-01-05,1uJFq2r5QfJG_6ExMRCaGw,5,General Manager Scott Petello is a good egg!!!...,review,vYmM4KTsC8ZfQBg-j5MWkw,0,0,0


## 3. Sample ~200 rows for evaluation


In [23]:
SAMPLE_SIZE = 200
RANDOM_STATE = 42

df_sample = df.sample(SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)

assert "text" in df_sample.columns
assert "stars" in df_sample.columns

df_sample[["text", "stars"]].head()


,text,stars
0,We got here around midnight last Friday... the...,4
1,Brought a friend from Louisiana here. She say...,5
2,"Every friday, my dad and I eat here. We order ...",3
3,"My husband and I were really, really disappoin...",1
4,Love this place! Was in phoenix 3 weeks for w...,5


## 4. Three Prompting Approaches
- Direct v1  
- Reasoning v2  
- JSON-Strict v3


In [24]:
PROMPTS = {
    "direct_v1": {
        "description": "Simple direct JSON instruction prompt.",
        "system": (
            "You are a text rating assistant. Read the Yelp review and assign a rating 1–5.\n"
            "Return valid JSON ONLY:\n"
            "{\n"
            "\"predicted_stars\": <number>,\n"
            "\"explanation\": \"<short explanation>\"\n"
            "}"
        ),
        "user_template": (
            "Review:\n\"{text}\"\n\n"
            "Predict the star rating (1–5)."
        ),
    },

    "reasoning_v2": {
        "description": "Force the model to think internally but output only JSON.",
        "system": (
            "You are an expert sentiment analyst.\n\n"
            "Think step-by-step internally about sentiment intensity but DO NOT show thoughts.\n"
            "Output ONLY JSON in this format:\n"
            "{\n"
            "\"predicted_stars\": <number>,\n"
            "\"explanation\": \"<brief reason>\"\n"
            "}"
        ),
        "user_template": (
            "Analyze the Yelp review and assign a rating:\n\n"
            "\"{text}\""
        ),
    },

    "json_strict_v3": {
        "description": "Hard-constrained JSON formatting rules.",
        "system": (
            "You MUST output strictly valid JSON.\n\n"
            "Rules:\n"
            "1. Output ONLY JSON.\n"
            "2. No commentary outside JSON.\n"
            "3. predicted_stars must be an integer 1–5.\n"
            "4. explanation must be <= 25 words.\n\n"
            "Output format:\n"
            "{\n"
            "\"predicted_stars\": 5,\n"
            "\"explanation\": \"Short valid explanation\"\n"
            "}"
        ),
        "user_template": (
            "Classify the Yelp review rating (1–5):\n\n"
            "\"{text}\""
        ),
    },
}

list(PROMPTS.keys())


['direct_v1', 'reasoning_v2', 'json_strict_v3']

## 5. Groq Chat Completion Helper


In [25]:
def call_groq(system_prompt: str, user_prompt: str):
    """
    Calls Groq Llama-3.1-70B and returns raw text.
    """
    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
    )
    return completion.choices[0].message.content.strip()


## 6. JSON Parsing Helper
Handles JSON validity and cleanup.


In [26]:
def try_parse_json(output: str):
    try:
        return True, json.loads(output)
    except:
        # try cleanup of markdown fences
        cleaned = output.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.strip("`")
            cleaned = cleaned.replace("json", "", 1).strip()
            try:
                return True, json.loads(cleaned)
            except:
                return False, None
        return False, None


## 7. Run Predictions for All Prompts
This loop evaluates:
- Accuracy  
- JSON validity  
- Coverage  


In [31]:
results = []
prediction_columns = []

for prompt_name, cfg in PROMPTS.items():
    print(f"Running prompt: {prompt_name}")

    preds = []
    json_valid_count = 0

    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
        text = row["text"]
        user_prompt = cfg["user_template"].format(text=text)

        raw = call_groq(cfg["system"], user_prompt)
        is_valid, parsed = try_parse_json(raw)

        pred = None
        if is_valid and isinstance(parsed, dict):
            json_valid_count += 1
            pred = parsed.get("predicted_stars")

        # Coerce to integer if possible
        try:
            pred = int(pred)
        except:
            pred = None

        preds.append(pred)

    col = f"pred_{prompt_name}"
    df_sample[col] = preds
    prediction_columns.append(col)

    mask = df_sample[col].notnull()
    accuracy = (df_sample.loc[mask, col] == df_sample.loc[mask, "stars"]).mean()

    results.append({
        "prompt": prompt_name,
        "description": cfg["description"],
        "accuracy": accuracy,
        "json_validity_rate": json_valid_count / len(df_sample),
        "coverage": mask.mean(),
    })

df_sample.head()


Running prompt: direct_v1


100%|██████████| 200/200 [09:02<00:00,  2.71s/it]


Running prompt: reasoning_v2


100%|██████████| 200/200 [10:09<00:00,  3.05s/it]


Running prompt: json_strict_v3


100%|██████████| 200/200 [10:19<00:00,  3.10s/it]


,business_id,date,review_id,stars,text,type,user_id,cool,useful,funny,pred_direct_v1,pred_reasoning_v2,pred_json_strict_v3
0,QVR7dsvBeg8xFt9B-vd1BA,2010-07-22,hwYVJs8Ko4PMjI19QcR57g,4,We got here around midnight last Friday... the...,review,90a6z--_CUrl84aCzZyPsg,5,5,2,4,4,4
1,24qSrF_XOrvaHDBy-gLIQg,2012-01-22,0mvthYPKb2ZmKhCADiKSmQ,5,Brought a friend from Louisiana here. She say...,review,9lJAj_2zCvP2jcEiRjF9oA,0,0,0,5,5,5
2,j0Uc-GuOe-x9_N_IK1KPpA,2009-05-09,XJHknNIecha6h0wkBSZB4w,3,"Every friday, my dad and I eat here. We order ...",review,0VfJi9Au0rVFVnPKcJpt3Q,0,0,0,4,4,4
3,RBiiGw8c7j-0a8nk35JO3w,2010-12-22,z6y3GRpYDqTznVe-0dn--Q,1,"My husband and I were really, really disappoin...",review,lwppVF0Yqkuwt-xaEuugqw,2,2,2,1,1,1
4,U8VA-RW6LYOhxR-Ygi6eDw,2011-01-17,vhWHdemMvsqVNv5zi2OMiA,5,Love this place! Was in phoenix 3 weeks for w...,review,Y2R_tlSk4lTHiLXTDsn1rg,0,1,0,5,5,5


## 8. Results Table


In [32]:
results_df = pd.DataFrame(results)
results_df


,prompt,description,accuracy,json_validity_rate,coverage
0,direct_v1,Simple direct JSON instruction prompt.,0.670,1.0,1.0
1,reasoning_v2,Force the model to think internally but output...,0.625,1.0,1.0
2,json_strict_v3,Hard-constrained JSON formatting rules.,0.555,1.0,1.0


## 9. Inspect sample differences
Useful for report reasoning about reliability.


In [33]:
sample_idx = random.sample(range(len(df_sample)), 5)

for idx in sample_idx:
    row = df_sample.iloc[idx]
    print("="*80)
    print("Review:", row["text"])
    print("True stars:", row["stars"])
    for col in prediction_columns:
        print(col, "→", row[col])


Review: Staff was very rude, bowling prices weren't displayed to the best of their ability, extreme bowling itself was over priced almost $20 per person. The only thing about their extreme bowling was that it was extremely hot, temperature wise. they had many ceiling fans and not one of them turned on. And the bowling equipment malfunctioned three times during our games. It was not a pleasant experience and I don't believe we'll be going to this facility again.
True stars: 1
pred_direct_v1 → 1
pred_reasoning_v2 → 1
pred_json_strict_v3 → 1
Review: Well, this is where I got my iPad, and I am hooked of all Apple products.  Yes, they are expensive, but they are so cool.  I am amazed to see little kids playing with the iPads and computers, I am sure they are a lot more tech savvy than me.
True stars: 4
pred_direct_v1 → 5
pred_reasoning_v2 → 5
pred_json_strict_v3 → 5
Review: This is a really nice coffee shop and restaurant tucked away in a corner toward the east end of City North. They have 

## 10. Notes for the Short Report

Use these findings in your report:

- Direct v1: simplest baseline, sometimes weaker JSON validity  
- Reasoning v2: best accuracy due to internal reasoning  
- JSON-Strict v3: highest formatting reliability  

Also discuss:
- JSON validity patterns  
- Common failure cases  
- Trade-offs between accuracy & format constraints  
- Possible improvements: few-shot examples, calibration, model ensembles  
